In [ ]:
import time
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
from influxdb_client.domain.write_precision import WritePrecision
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ─── Config ───
url    = "http://192.168.137.106:8086"
token  = "xzQO7Slj8SDaHBVfpnq8AYJ14Bq3NM20ZpWiZ28KYLboK2eH70uJZ15AyNNuI2G_e_OMMh0MDR-jIGTHJxye9A=="
org    = "CUEE2102541"
bucket = "ESP32"
INTERVAL = 1 * 60  # รันทุก 1 นาที (ปรับได้)

client    = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()
write_api = client.write_api(write_options=SYNCHRONOUS)
FEATURES  = ["temp", "humi", "pres", "accel_mag", "gyro_mag"]

# ════════════════════════════════════════
# STEP 1: Train ครั้งเดียวด้วยข้อมูล 24h
# ════════════════════════════════════════
print("📚 กำลัง Train โมเดล...")

train_query = '''
from(bucket: "ESP32")
  |> range(start: -24h)
  |> filter(fn: (r) => r._measurement == "IoTFun_final")
  |> pivot(
      rowKey: ["_time", "node_name", "device_id"],
      columnKey: ["_field"],
      valueColumn: "_value"
     )
  |> sort(columns: ["_time"])
'''

tables = query_api.query(train_query)
rows = []
for table in tables:
    for record in table.records:
        rows.append(record.values)

df_train = pd.DataFrame(rows)
df_train = df_train.drop(columns=["result","table","_start","_stop","_measurement"], errors="ignore")
df_train["accel_mag"] = np.sqrt(df_train["acx"]**2 + df_train["acy"]**2 + df_train["acz"]**2)
df_train["gyro_mag"]  = np.sqrt(df_train["gyx"]**2 + df_train["gyy"]**2 + df_train["gyz"]**2)

scaler     = StandardScaler()
X_train    = scaler.fit_transform(df_train[FEATURES])
iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso_forest.fit(X_train)

print(f"✅ Train เสร็จด้วยข้อมูล {len(df_train)} แถว")

# ════════════════════════════════════════
# STEP 2: Loop Predict ทุก 1 นาที
# ════════════════════════════════════════
def run_pipeline():
    print(f"\n🔄 [{datetime.now().strftime('%H:%M:%S')}] กำลังรัน pipeline...")

    # ดึงข้อมูล 1 นาทีล่าสุด
    query = '''
    from(bucket: "ESP32")
      |> range(start: -1m)
      |> filter(fn: (r) => r._measurement == "IoTFun_final")
      |> pivot(
          rowKey: ["_time", "node_name", "device_id"],
          columnKey: ["_field"],
          valueColumn: "_value"
         )
      |> sort(columns: ["_time"])
    '''
    tables = query_api.query(query)
    rows = []
    for table in tables:
        for record in table.records:
            rows.append(record.values)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        print("⚠️ ไม่มีข้อมูลในช่วงนี้ ข้ามรอบนี้ไป")
        return

    df = df.drop(columns=["result","table","_start","_stop","_measurement"], errors="ignore")
    df["accel_mag"] = np.sqrt(df["acx"]**2 + df["acy"]**2 + df["acz"]**2)
    df["gyro_mag"]  = np.sqrt(df["gyx"]**2 + df["gyy"]**2 + df["gyz"]**2)

    # Predict (ไม่ train ใหม่)
    X_scaled            = scaler.transform(df[FEATURES])
    df["anomaly_score"] = iso_forest.decision_function(X_scaled)
    df["anomaly"]       = iso_forest.predict(X_scaled)

    # สรุปรายกล่อง
    summary = df.groupby("node_name").agg(
        total     = ("anomaly", "count"),
        spoiled   = ("anomaly", lambda x: (x == -1).sum()),
        avg_temp  = ("temp", "mean"),
        avg_humi  = ("humi", "mean"),
        avg_accel = ("accel_mag", "mean"),
        min_score = ("anomaly_score", "min"),
    ).reset_index()

    summary["spoiled_pct"] = (summary["spoiled"] / summary["total"] * 100).round(1)
    summary["verdict"] = summary["spoiled_pct"].apply(
        lambda x: "❌ เสีย" if x > 20 else ("⚠️ เฝ้าระวัง" if x > 5 else "✅ ปลอดภัย")
    )

    # ส่งกลับ InfluxDB
    now = datetime.now(timezone.utc)
    for _, row in summary.iterrows():
        status_code = {"✅ ปลอดภัย": 0, "⚠️ เฝ้าระวัง": 1, "❌ เสีย": 2}.get(row["verdict"], -1)
        point = (
            Point("drug_status")
                .tag("node_name",     row["node_name"])
                .field("status_code",   status_code)
                .field("status_label",  row["verdict"])
                .field("spoiled_pct",   float(row["spoiled_pct"]))
                .field("avg_temp",      float(row["avg_temp"]))
                .field("avg_humi",      float(row["avg_humi"]))
                .field("avg_accel",     float(row["avg_accel"]))
                .field("anomaly_score", float(row["min_score"]))
                .time(now, WritePrecision.NS)
        )
        write_api.write(bucket=bucket, org=org, record=point)

    print(f"✅ ส่งข้อมูลเรียบร้อย")
    for _, row in summary.iterrows():
        print(f"   {row['node_name']} → {row['verdict']} (temp={row['avg_temp']:.1f}°C, humi={row['avg_humi']:.1f}%)")

# ════════════════════════════════════════
# START
# ════════════════════════════════════════
print(f"\n🚀 เริ่ม pipeline อัตโนมัติทุก {INTERVAL//60} นาที")
print("⛔ กด Kernel → Interrupt Kernel เพื่อหยุด\n")

while True:
    run_pipeline()
    print(f"⏳ รอ {INTERVAL//60} นาที...")
    time.sleep(INTERVAL)

📚 กำลัง Train โมเดล...
✅ Train เสร็จด้วยข้อมูล 14172 แถว

🚀 เริ่ม pipeline อัตโนมัติทุก 1 นาที
⛔ กด Kernel → Interrupt Kernel เพื่อหยุด


🔄 [16:08:19] กำลังรัน pipeline...
✅ ส่งข้อมูลเรียบร้อย
   Node_01 → ✅ ปลอดภัย (temp=36.4°C, humi=48.9%)
   Node_02 → ❌ เสีย (temp=34.9°C, humi=51.8%)
   Node_03 → ✅ ปลอดภัย (temp=33.9°C, humi=52.9%)
⏳ รอ 1 นาที...

🔄 [16:09:19] กำลังรัน pipeline...
✅ ส่งข้อมูลเรียบร้อย
   Node_01 → ✅ ปลอดภัย (temp=36.6°C, humi=48.2%)
   Node_02 → ❌ เสีย (temp=35.1°C, humi=51.3%)
   Node_03 → ✅ ปลอดภัย (temp=33.9°C, humi=53.1%)
⏳ รอ 1 นาที...

🔄 [16:10:19] กำลังรัน pipeline...
✅ ส่งข้อมูลเรียบร้อย
   Node_01 → ✅ ปลอดภัย (temp=37.1°C, humi=47.5%)
   Node_02 → ❌ เสีย (temp=34.9°C, humi=51.5%)
   Node_03 → ✅ ปลอดภัย (temp=34.1°C, humi=52.5%)
⏳ รอ 1 นาที...

🔄 [16:11:19] กำลังรัน pipeline...
✅ ส่งข้อมูลเรียบร้อย
   Node_01 → ⚠️ เฝ้าระวัง (temp=37.5°C, humi=46.5%)
   Node_02 → ❌ เสีย (temp=35.1°C, humi=51.0%)
   Node_03 → ✅ ปลอดภัย (temp=34.0°C, humi=52.6%)
⏳ รอ 1 นาที...
